In [ ]:
# Inicio de actividad 2 Hito 4

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# -*- coding: utf-8 -*-
"""
motor_orquestador_biovid_v2.py
==============================
Pipeline Optimizado: Extracción Vectorizada de Bioseñales y Normalización Z-Score.

Arquitectura mejorada para alta velocidad I/O y procesamiento en memoria.
INSTRUCCIONES:
- Prepara las bibliotecas correspondientes, especialmente: neurokit2
- Verifica las rutas y los nombres de las mismas
- Ejecuta y ya esta
"""

import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
from tqdm import tqdm
import sys
import sys
import os
ruta_modulo = '/content/drive/Shareddrives/DATA_GPI/Grupo1_Desarrollo/Entregable Grupo Uno/Hito 4/Actividad 2'
sys.path.append(ruta_modulo)

import signal_processor_V2 as sp  # Requiere v2.0.0

# --- CONFIGURACIÓN DE RUTAS ---
BASE_DIR = Path('/content/drive/Shareddrives/GPI')
ZIP_SEÑALES = BASE_DIR / 'biosignals_filtered.zip'
CSV_SAMPLES = BASE_DIR / 'samples.csv'
CSV_FINAL = Path('/content/drive/Shareddrives/DATA_GPI/Datasets Normalizados/dataset_features_biovid_normalizado_V2.csv')

def procesar_extraccion_masiva() -> pd.DataFrame:
    print("\n[+] INICIANDO FASE 1: EXTRACCIÓN OPTIMIZADA DE CARACTERÍSTICAS")

    if not ZIP_SEÑALES.exists():
        raise FileNotFoundError(f"Archivo no detectado: {ZIP_SEÑALES}")

    df_maestro = pd.read_csv(CSV_SAMPLES, sep='\t')
    resultados = []

    with zipfile.ZipFile(ZIP_SEÑALES, 'r') as zip_ref:
        # Búsqueda O(1) de archivos para máxima velocidad
        archivos_disponibles = set(zip_ref.namelist())

        # itertuples() es sustancialmente más rápido que iterrows()
        for row in tqdm(df_maestro.itertuples(index=False), total=len(df_maestro), desc="Analizando señales"):
            info_paciente = row._asdict()
            ruta_estricta = f"biosignals_filtered/{row.subject_name}/{row.sample_name}_bio.csv"

            if ruta_estricta not in archivos_disponibles:
                info_paciente['estado_extraccion'] = "FALLO_NO_ENCONTRADO"
                resultados.append(info_paciente)
                continue

            try:
                with zip_ref.open(ruta_estricta) as f:
                    df_signal = pd.read_csv(f, sep='\t')

                # Inyección dinámica de métricas
                if 'ecg' in df_signal.columns:
                    info_paciente.update(sp.extract_ecg_features(df_signal['ecg'].values))

                if 'gsr' in df_signal.columns:
                    info_paciente.update(sp.extract_gsr_features(df_signal['gsr'].values))

                for col_emg in ['emg_trapezius', 'emg_corrugator', 'emg_zygomaticus']:
                    if col_emg in df_signal.columns:
                        feats = sp.extract_emg_features(df_signal[col_emg].values)
                        info_paciente.update({f"{col_emg}_{k.replace('emg_', '')}": v for k, v in feats.items()})

                info_paciente['estado_extraccion'] = "OK"

            except Exception as e:
                info_paciente['estado_extraccion'] = f"ERROR: {str(e)}"

            resultados.append(info_paciente)

    return pd.DataFrame(resultados).fillna(0.0)

def aplicar_zscore_vectorizado(df_raw: pd.DataFrame) -> pd.DataFrame:
    print("\n[+] INICIANDO FASE 2: NORMALIZACIÓN Z-SCORE (VECTORIZADA)")

    # Filtrar solo métricas numéricas excluyendo identificadores
    cols_ignorar = {'subject_id', 'subject_name', 'class_id', 'class_name', 'sample_id', 'sample_name', 'estado_extraccion'}
    features = [c for c in df_raw.columns if c not in cols_ignorar and pd.api.types.is_numeric_dtype(df_raw[c])]

    print(f"[*] Normalizando {len(features)} características por sujeto...")

    # Función vectorizada con numpy para evitar el lento iterador de python
    def z_score_numpy(x):
        std = x.std(ddof=0)
        # np.where actúa como un if/else instantáneo a nivel de tensores
        return np.where(std > 0, (x - x.mean()) / std, 0.0)

    # El uso de transform() preserva el índice y es altamente superior a apply()
    df_raw[features] = df_raw.groupby('subject_name', group_keys=False)[features].transform(z_score_numpy)

    return df_raw

if __name__ == "__main__":
    try:
        df_crudo = procesar_extraccion_masiva()

        # Filtramos posibles errores de lectura antes de normalizar
        df_limpio = df_crudo[df_crudo['estado_extraccion'] == "OK"].copy()

        if df_limpio.empty:
            print("[-] Abortando: No se extrajeron datos válidos.")
        else:
            df_final = aplicar_zscore_vectorizado(df_limpio)

            # Limpiar columna de control interno antes de entrega
            df_final.drop(columns=['estado_extraccion'], inplace=True, errors='ignore')

            df_final.to_csv(CSV_FINAL, index=False, float_format='%.6f', decimal='.')
            print(f"\n[✓] MISIÓN CUMPLIDA. Dataset inmaculado listo en: {CSV_FINAL}")

    except Exception as e:
        print(f"\n[!] FALLO CRÍTICO EN EL SISTEMA: {e}")


[+] INICIANDO FASE 1: EXTRACCIÓN OPTIMIZADA DE CARACTERÍSTICAS


Analizando señales:  36%|███▋      | 3137/8600 [15:26<02:59, 30.48it/s]/usr/local/lib/python3.12/dist-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
Analizando señales:  37%|███▋      | 3145/8600 [15:26<03:12, 28.35it/s]/usr/local/lib/python3.12/dist-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
Analizando señales:  37%|███▋      | 3166/8600 [15:27<03:31, 25.71it/s]/usr/local/lib/python3.12/dist-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
Analizando señales:  48%|████▊ 


[+] INICIANDO FASE 2: NORMALIZACIÓN Z-SCORE (VECTORIZADA)
[*] Normalizando 15 características por sujeto...

[✓] MISIÓN CUMPLIDA. Dataset inmaculado listo en: /content/drive/Shareddrives/DATA_GPI/Datasets Normalizados/dataset_features_biovid_normalizado_V2.csv
